***Задачи_Семинар_3***

In [ ]:
'''
Задача 2. Аудит доступов
Служба безопасности сравнивает вчерашнюю и сегодняшнюю выгрузку сотрудников с доступом к продакшену.

Дано:

yesterday = {"anna", "boris", "clara", "dmitry", "elena", "fedor"}
today = {"boris", "clara", "elena", "george", "hanna"}
Нужно (все результаты — отсортированные списки, чтобы отчёт был воспроизводим):

granted — кому доступ выдали;
revoked — у кого отозвали;
unchanged — у кого доступ остался;
changed_count — сколько всего сотрудников попало в изменения (в обе стороны);
alert — строка Доступы: +2 / -3 (выдано / отозвано).
'''
yesterday = {"anna", "boris", "clara", "dmitry", "elena", "fedor"}
today = {"boris", "clara", "elena", "george", "hanna"}


print("granted:",sorted(today-yesterday))
print("revoked:",sorted(yesterday-today))
print("unchanged:",sorted(today&yesterday))
print("changed_count:",len(granted)+len(revoked))
print(f'alert Доступы: +{len(granted)} / -{len(revoked)} (выдано / отозвано)')


In [ ]:
'''
Задача 3. Кэш геокодера на кортежах
Сервис доставки кэширует результаты платного геокодера: по паре (город, улица) хранится координата.

Нужно:

создать словарь geo_cache с двумя записями:
("Москва", "Тверская") → (55.7649, 37.6053)
("Казань", "Баумана") → (55.7903, 49.1221)
moscow_lat — широта Тверской (число, через распаковку кортежа);
is_cached — есть ли в кэше ("Казань", "Баумана");
добавить ("Сочи", "Навагинская") → (43.5855, 39.7231) и получить cache_size;
miss — результат запроса ("Пермь", "Ленина") через .get() со значением по умолчанию None.
'''
geo_cache = {("Москва", "Тверская"):(55.7649, 37.6053),("Казань", "Баумана"):(55.7903, 49.1221)}
moscow_lat,_ = geo_cache[("Москва", "Тверская")]
print(moscow_lat)
is_cached = ("Казань", "Баумана") in geo_cache
print(is_cached)
# print(geo_cache.update({("Сочи", "Навагинская"):(43.5855, 39.7231)}))
# print(geo_cache)
geo_cache[("Сочи", "Навагинская")] = (43.5855, 39.7231)
print(geo_cache)
cache_size = len(geo_cache)
miss = geo_cache.get(("Пермь", "Ленина"), None)
#print(miss)


In [ ]:
'''
Задача 4. Разбор ответа API
Платёжный шлюз вернул JSON (в Python это уже словарь).

Нужно достать/посчитать:

total_records — всего записей по метаданным;
first_user — имя пользователя в первой записи;
amounts — список сумм всех трёх записей (обращаемся по индексам);
total_amount — сумма всех сумм;
max_amount — максимальная сумма;
currency — валюта из meta; ключа там нет, поэтому через .get() с дефолтом "RUB";
has_next_page — True, если page * per_page < total.
'''
response = {
    "status": "ok",
    "meta": {"page": 2, "per_page": 50, "total": 137},
    "data": [
        {"id": 101, "user": "anna", "amount": 1250.0, "status": "paid"},
        {"id": 102, "user": "boris", "amount": 890.5, "status": "pending"},
        {"id": 103, "user": "clara", "amount": 4300.0, "status": "paid"},
    ],
}


# total_records = response["meta"]["total"]
# print(total_records)

# first_user = response["data"][0]["user"]
# print(first_user)

# a = response["data"][0]["amount"]
# print(a)
# b = response["data"][1]["amount"]
# print(b)
# c = response["data"][2]["amount"]
# print(c)
# amounts = [a, b,c]

# total_amount = a+b+c

# max_amount = max(a,b,c)

# currency = response["meta"].get("currency", "rub")

has_next_page = response["meta"]["page"]*response["meta"]["per_page"]<response["meta"]["total"]
print(has_next_page)

# a, b, c = response["meta"].keys()
# print(a, b, c, sep="\n")
# has_next_page == True: page * per_page < total

print(total_records, first_user, amounts, total_amount, max_amount, currency, has_next_page, sep=" | ")



In [ ]:
'''
Задача 5. Слияние конфигурации
В приложении настройки собираются из трёх источников с ростом приоритета: значения по умолчанию → переменные окружения → аргументы командной строки.

Нужно:

final_config — итоговый словарь (используйте оператор | или update на копии);
overridden — отсортированный список ключей, значение которых отличается от значения по умолчанию;
report — строка Переопределено: debug, host, timeout.
Важно: defaults не должен измениться — проверка это контролирует.
'''

defaults = {"host": "localhost", "port": 5432, "timeout": 30, "debug": False}
env = {"host": "db-prod.local", "timeout": 5}
cli = {"debug": True}

# TODO
final_config = defaults.copy()
final_config.update(env)
final_config.update(cli)
overridden = sorted(final_config.keys())
print(overridden)

# overridden = ...
# report = ...

# print(final_config)
# print(report)


In [ ]:
'''
Задача 6 (со звёздочкой). Перцентили латентности
Нагрузочное тестирование вернуло 20 замеров времени ответа (мс). Среднее в мониторинге бесполезно — дежурному нужны медиана и p95.

Нужно:

sorted_latencies — отсортированный список;
avg — среднее, округлить до 1 знака;
median — медиана (у чётного количества — среднее двух центральных);
p95 — значение по «методу ближайшего ранга»: индекс ceil(0.95 * n) - 1 в отсортированном списке;
verdict — строка avg=221.4 median=155.0 p95=420 .
Обратите внимание, насколько среднее «врёт» из-за одного выброса в 990 мс.
'''

import math

latencies = [120, 98, 340, 210, 150, 95, 420, 180, 130, 160, 175, 88, 990, 205, 145, 132, 118, 260, 310, 102]

# TODO

sorted_latencies = sorted(latencies)
print(sorted_latencies)

import statistics
avg = round(statistics.mean(latencies), 1)
print(avg)

median = statistics.median(latencies)
print(median)

import math
n = len(sorted_latencies)
index = math.ceil(0.95 * n) - 1
p95 = sorted_latencies[index]
print(p95)

verdict = f'avg{avg}, median={median}, p95={p95}'

print(verdict)